# 📊 HR Analytics Data Processing & Feature Engineering Pipeline

## Overview

This notebook presents an end-to-end **HR analytics data processing pipeline** designed to transform raw employee data into a structured, analysis-ready dataset for downstream use in **SQL (PostgreSQL), data modeling, and Power BI dashboarding**.

The dataset contains employee-level HR records, including demographic information, employment history, salary data, and performance-related attributes. The primary goal is to ensure the data is **cleaned, standardized, validated, and enriched with meaningful business metrics** that support workforce analytics and decision-making.

---

## 🎯 Objectives

This pipeline focuses on the following key objectives:

### 1. Data Standardization
- Normalize column naming conventions for SQL compatibility  
- Ensure consistent and machine-readable schema formatting  

### 2. Data Quality & Cleaning
- Handle missing values and inconsistent data types  
- Remove duplicate employee records  
- Standardize string and numeric formats  

### 3. Feature Engineering
- Compute employee tenure (active and total lifecycle)  
- Create tenure-based segmentation for workforce analysis  
- Generate compensation metrics such as compa-ratio  
- Derive statistical features for salary distribution analysis  

### 4. Data Validation & Quality Checks
- Detect anomalies in age vs experience relationships  
- Identify salary outliers using statistical methods (z-score)  
- Flag inconsistencies in job level compensation expectations  
- Classify missing or incomplete performance data  

### 5. Analytics Readiness
- Prepare dataset for relational database storage (PostgreSQL)  
- Enable transformation into SQL views and fact/dimension models  
- Support visualization in BI tools such as Power BI  

---

## 📦 Output

The final output of this notebook is a **fully enriched HR analytics dataset** containing:

- Cleaned and standardized employee attributes  
- Derived workforce metrics (tenure, salary ratios, etc.)  
- Performance classification logic  
- Data quality and anomaly detection flags  

This dataset is intended to serve as a **gold-layer analytics table**, suitable for:

- SQL-based reporting layers  
- BI dashboard development  
- Workforce analytics insights  

---

## ⚙️ Technical Context

This pipeline is designed with scalability and production alignment in mind, using:

- Vectorized operations for large-scale datasets (millions of rows)  
- SQL-mappable transformations (groupby, aggregation logic, filtering rules)  
- Audit-safe design principles (non-destructive feature engineering)  
- Modular transformation stages aligned with ETL best practices  

---

In [27]:
# ==========================================
# IMPORTS
# ==========================================
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine
import urllib.parse

In [28]:
# ==========================================
# PIPELINE METADATA (LINEAGE TRACKING)
# ==========================================
# In production HR systems, every transformation must be traceable.
# This helps with audits, debugging, and reproducibility of analytics.

PIPELINE_VERSION = "1.0.0"
LOAD_TIMESTAMP = datetime.now()

In [ ]:
# ==========================================
# DATABASE CONNECTION SETUP
# ==========================================

password = "***@***" # Change this
safe_password = urllib.parse.quote_plus(password)
user = "*******" 
host = "*******"
port = "*******"
db_name = "emp_record_db"

connection_string = f"postgresql://{user}:{safe_password}@{host}:{port}/{db_name}"
engine = create_engine(connection_string)

In [30]:
# ==========================================
# RAW DATA INGESTION (BRONZE LAYER)
# ==========================================
# We immediately copy the raw dataset to preserve a "source of truth".
# This is critical in enterprise HR systems where raw data must never be overwritten
# because it may be used for audits, compliance checks, or historical reconstruction.

df_raw = pd.read_csv(r'C:\Users\Kim Singson\OneDrive\Desktop\Data Projects\HR Dataset\hr_raw.csv')
df = df_raw.copy()

In [31]:
# ==========================================
# SCHEMA STANDARDIZATION
# ==========================================
# Column names are normalized to lowercase + underscores to:
# - prevent case-sensitive errors
# - ensure compatibility with SQL (PostgreSQL best practice)

df.columns = df.columns.str.strip().str.lower()

In [32]:
# ==========================================
# COLUMN CONTRACT FIXES (RESILIENCE LAYER)
# ==========================================
# HR systems often change column names across exports (e.g., HRIS updates).
# This step ensures the pipeline does not break due to naming inconsistencies.

if 'experience_years' in df.columns:
    df = df.rename(columns={'experience_years': 'prev_exp_year'})

# If missing entirely, we create a safe null column to avoid pipeline failure.
# This is important for large-scale pipelines where schema drift is common.
if 'prev_exp_year' not in df.columns:
    df['prev_exp_year'] = np.nan

In [33]:
# ==========================================
# DATA TYPE SAFETY CONVERSION
# ==========================================
# Explicit typing prevents silent errors during calculations at scale.

df['hire_date'] = pd.to_datetime(df['hire_date'], errors='coerce')

df['salary'] = pd.to_numeric(df['salary'], errors='coerce')

numeric_cols = ['prev_exp_year', 'year', 'age']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [34]:
# ==========================================
# TEXT CLEANING (SAFE VERSION)
# ==========================================
# String cleaning ensures that duplicates caused by spacing/case inconsistencies
# do not inflate counts in analytics (very common HR data issue).

text_cols = [
    'employee_id','full_name','department','job_title',
    'performance_rating','status','work_mode',
    'country','city','job_level'
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

In [35]:
# ==========================================
# DATA QUALITY FLAGS (RAW LAYER)
# ==========================================

# Missing salary is critical because compensation is a core HR metric.
df['missing_salary_flag'] = df['salary'].isna()

# Missing performance ratings may indicate process gaps in HR evaluation cycles.
df['missing_performance_flag'] = df['performance_rating'].isna()

# We use 16 as a conservative global heuristic.
# This is a legal/compliance check. 

# If an employee is 14, it doesn't matter how much experience they have; 
# in many regions, that is a labor law violation. It’s a "Hard Error."

df['invalid_age_flag'] = df['age'] < 16

# Duplicate detection is important because HR systems often re-import employees
# due to system merges, rehires, or HRIS sync issues.
df['duplicate_employee_flag'] = df.duplicated(subset=['employee_id'], keep=False)

In [36]:
# ==========================================
# 🛡️ HR ARCHITECTURE SAFEGUARD
# ==========================================
# We NEVER deduplicate here.
# HR data is EVENT-BASED, not snapshot-based.
# Removing duplicates too early destroys historical employment patterns.

df_history = df.copy()

In [37]:
# ==========================================
# CURRENT SNAPSHOT (ONLY PLACE WHERE DEDUP OCCURS)
# ==========================================
# This creates a "current state view" used for dashboards.
# We keep only the latest record per employee based on hire_date.

df_current = (
    df_history.sort_values(['employee_id', 'hire_date'], ascending=[True, False])
              .drop_duplicates(subset=['employee_id'], keep='first')
              .copy()
)

In [38]:
# ==========================================
# FEATURE ENGINEERING BASE DATASET
# ==========================================
# We now work on full history to preserve analytical completeness.

df = df_history.copy()

current_date = pd.to_datetime(datetime.now())

In [39]:
# ==========================================
# TENURE ENGINEERING
# ==========================================
# 365.25 is used instead of 365 to account for leap years,
# improving long-term accuracy (important for enterprise datasets).

df['tenure_total'] = (
    (current_date - df['hire_date']).dt.days / 365.25
).round(2)

# Active tenure is only meaningful for current employees.
df['tenure_active'] = np.where(
    df['status'].str.lower() == 'active',
    df['tenure_total'],
    np.nan
)

In [40]:
# ==========================================
# TENURE SEGMENTATION (HR LIFECYCLE MODEL)
# ==========================================
# These bins represent standard HR lifecycle stages:
# - <3 months: onboarding phase
# - <1 year: early adaptation
# - 1–3 years: stabilization
# - 3–5 years: experienced employees
# - 5+ years: senior workforce

# These thresholds are commonly used in HR analytics to study retention behavior.

bins = [0, 0.25, 1, 3, 5, np.inf]
labels = ['New (<3m)', 'Early', 'Junior', 'Mid', 'Senior']

df['tenure_band'] = pd.cut(df['tenure_total'], bins=bins, labels=labels)

In [41]:
# ==========================================
# PERFORMANCE HANDLING
# ==========================================
# We keep BOTH raw and cleaned versions:
# - raw is for audit / traceability
# - clean is for analytics consistency

df['performance_rating_raw'] = df['performance_rating']

# Missing ratings are converted to "Unrated" because BI tools
# handle categorical completeness better than NULL values.
df['performance_rating_clean'] = df['performance_rating'].fillna("Unrated")

df['performance_missing_flag'] = df['performance_rating'].isna()

In [42]:
# ==========================================
# COMPENSATION ENGINEERING (PAY EQUITY MODEL)
# ==========================================
# Salary is often skewed (few high earners distort averages),
# so we use MEDIAN instead of mean for fairness benchmarking.

df['salary'] = df['salary'].fillna(0)

valid_salary = df['salary'] > 0

benchmark = (
    df[valid_salary]
    .groupby(['department','job_level'])['salary']
    .median()
    .reset_index()
    .rename(columns={'salary':'benchmark_salary'})
)

df = df.merge(benchmark, on=['department','job_level'], how='left')

# Compa-ratio is a standard HR metric:
# 1.0 = aligned with internal pay benchmark
# <0.8 = potential underpayment risk (commonly flagged in HR audits)
# >1.2 = potential overpayment or role misalignment
df['compa_ratio'] = np.where(
    df['salary'] > 0,
    df['salary'] / df['benchmark_salary'].replace(0, np.nan),
    np.nan
)

df['compa_flag'] = np.select(
    [
        df['compa_ratio'] < 0.8,
        df['compa_ratio'] > 1.2
    ],
    ['Underpaid', 'Overpaid'],
    default='Fair'
)

In [43]:
# ==========================================
# STATISTICAL OUTLIERS (Z-SCORE METHOD)
# ==========================================
# Z-score measures how far a value is from the group mean in standard deviations.
# Threshold ±3 is used because in a normal distribution,
# ~99.7% of values lie within this range.

# This is widely used for anomaly detection in large datasets.

df['group_mean_salary'] = df.groupby(['department','job_level'])['salary'].transform('mean')
df['group_std_salary'] = df.groupby(['department','job_level'])['salary'].transform('std')

df['salary_zscore'] = (
    (df['salary'] - df['group_mean_salary']) /
    df['group_std_salary']
)

df['salary_outlier_flag'] = np.where(
    df['salary_zscore'].abs() > 3,
    'Outlier',
    'Normal'
)

In [44]:
# ==========================================
# DATA QUALITY SCORE (COMPOSITE HEALTH METRIC)
# ==========================================
# Instead of only flags, we build a score:
# Each issue contributes 1 point.
# Higher score = lower data reliability.

df['data_quality_score'] = (
    df['missing_salary_flag'].astype(int) +
    df['missing_performance_flag'].astype(int) +
    df['invalid_age_flag'].astype(int) +
    df['duplicate_employee_flag'].astype(int)
)

# This helps executives quickly segment:
# - clean vs unreliable HR data
df['data_quality_flag'] = np.select(
    [
        df['data_quality_score'] >= 3,
        df['data_quality_score'] == 2
    ],
    ['Low Quality', 'Medium Quality'],
    default='High Quality'
)

In [45]:
# ==========================================
# ATTRITION RISK MODEL (HEURISTIC SCORING)
# ==========================================
# This is NOT machine learning.
# It is a rule-based scoring system commonly used in HR dashboards.

# Each factor adds 1 risk point:
# - low compa_ratio → financial dissatisfaction risk
# - missing performance → weak HR engagement signal
# - salary outlier → abnormal compensation behavior
# - long tenure (>5 years) → possible stagnation risk

df['attrition_risk_score'] = (
    (df['compa_ratio'] < 0.8).astype(int) +
    (df['performance_missing_flag']).astype(int) +
    (df['salary_outlier_flag'] == 'Outlier').astype(int) +
    (df['tenure_total'] > 5).astype(int)
)

# Risk categories are derived from cumulative score:
# - 3+ signals = high likelihood of attrition risk
# - 2 signals = moderate risk requiring monitoring
df['attrition_risk_flag'] = np.select(
    [
        df['attrition_risk_score'] >= 3,
        df['attrition_risk_score'] == 2
    ],
    ['High Risk', 'Medium Risk'],
    default='Low Risk'
)


In [46]:
# ==========================================
# SALARY POSITIONING & JOB VALIDATION
# ==========================================
# Calculates where an employee sits relative to others in the same group
df['salary_percentile'] = df.groupby(['department', 'job_level'])['salary'].rank(pct=True).round(4)

# Flag for salaries in the bottom 10% of their group
df['low_pay_flag'] = np.where(df['salary_percentile'] < 0.1, 'Low Pay', 'Standard')

# Replicating "Level Mismatch" - if salary is 0 or very low for a high job level
df['level_expected_salary'] = df['benchmark_salary']  # Using median as the expectation
df['level_mismatch_flag'] = np.where(
    (df['job_level'].isin(['Senior', 'Director'])) & (df['compa_ratio'] < 0.6),
    'Mismatch: Underpaid for Level',
    'Aligned'
)

In [47]:
# ==========================================
# ORGANIZATIONAL STRUCTURE
# ==========================================
# Count total employees per department
df['department_headcount'] = df.groupby('department')['employee_id'].transform('count')

# Count how many people share the same job level within that department
df['job_level_density'] = df.groupby(['department', 'job_level'])['employee_id'].transform('count')

In [48]:
# ==========================================
# AGE + EXPERIENCE VALIDATION
# ==========================================
# Heuristic: Minimum logical age = 18 + years of experience
# This is a data integrity check. 

# If someone is 22 years old but the record says they have 10 years of experience, 
# they would have had to start working at age 12. 
# While they pass the "Age > 16" rule, the combination of age and experience 
# is physically or logically improbable. This is a "Soft Error" (likely a typo).

df['expected_age'] = 18 + df['prev_exp_year'].fillna(0)

df['age_validation_flag'] = np.select(
    [
        (df['invalid_age_flag'] == True),                   # Priority 1: Legal violation
        (df['age'] < df['expected_age'])                    # Priority 2: Logical impossibility
    ],
    [
        'Invalid: Underage', 
        'Potential Data Entry Error (Age vs Exp)'
    ],
    default='Valid'
)

In [49]:
# ==========================================
# PIPELINE LINEAGE TRACKING
# ==========================================
# This ensures traceability in production systems:
# You always know WHEN and WHICH version generated the dataset.

df['pipeline_version'] = PIPELINE_VERSION
df['load_timestamp'] = LOAD_TIMESTAMP


In [50]:
# ==========================================
# COLUMN ORDERING (READABILITY ONLY)
# ==========================================
# This does NOT affect computation.
# It improves:
# - human readability
# - debugging
# - BI tool usability

ordered_cols = [
    "employee_id", "full_name", "department", "job_title", "job_level",
    "country", "city", "work_mode", "hire_date", "status", "year", "age",
    "prev_exp_year", "salary",
    
    # Performance
    "performance_rating_raw", "performance_rating_clean", "performance_missing_flag",
    
    # Tenure
    "tenure_total", "tenure_active", "tenure_band",
    
    # Comp Engineering
    "benchmark_salary", "compa_ratio", "compa_flag",
    
    # Stats
    "group_mean_salary", "group_std_salary", "salary_zscore", "salary_outlier_flag",
    
    # Positioning (New)
    "salary_percentile", "low_pay_flag",
    
    # Level Validation (New)
    "level_expected_salary", "level_mismatch_flag",
    
    # Org Structure (New)
    "department_headcount", "job_level_density",
    
    # Age Validation (New)
    "expected_age", "age_validation_flag",
    
    # Quality & Risk
    "missing_salary_flag", "missing_performance_flag", "invalid_age_flag",
    "duplicate_employee_flag", "data_quality_score", "data_quality_flag",
    "attrition_risk_score", "attrition_risk_flag",
    
    # Lineage
    "pipeline_version", "load_timestamp"
]

df = df[[col for col in ordered_cols if col in df.columns]]

In [51]:
# ==========================================
# OUTPUT LAYERS (PRODUCTION STRUCTURE)
# ==========================================
# We separate outputs into layers:
# - gold: analytics-ready dataset
# - history: full audit trail (never modified)
# - snapshot: current workforce view

df_gold = df.copy()
df_history_final = df_history.copy()
df_snapshot = df_current

In [56]:
df_gold.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 45 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   employee_id               string        
 1   full_name                 string        
 2   department                string        
 3   job_title                 string        
 4   job_level                 string        
 5   country                   string        
 6   city                      string        
 7   work_mode                 string        
 8   hire_date                 datetime64[ns]
 9   status                    string        
 10  year                      int64         
 11  age                       int64         
 12  prev_exp_year             int64         
 13  salary                    float64       
 14  performance_rating_raw    string        
 15  performance_rating_clean  string        
 16  performance_missing_flag  bool          
 17  tenure_t

In [57]:
# ==========================================
# LOAD TO pgAdmin PostgreSQL
# ===========================================

schema_name = 'hr_record'

try:
    # chunksize helps manage memory for 2M rows
    df_gold.to_sql(schema_name, engine, if_exists='replace', index=False, chunksize=10000)
    print(f"ETL Complete: {len(df):,} unique records successfully loaded to PostgreSQL.")
except Exception as e:
    print(f"Database Load Error: {e}")


ETL Complete: 2,000,000 unique records successfully loaded to PostgreSQL.
